# 09: Production Model Refit, CLI Inference & Farmer Profit Margin

### Notebook Purpose & Research Objectives
This notebook demonstrates the end-to-end deployment workflow:
1. Verifying the selected best model (`simple_average_top3`) and reviewing its model card.
2. Executing the CLI inference script (`predict.py`) for arbitrary vegetables and forecast horizons.
3. Combining market price forecasts with crop production costs to compute farmer profit margins with 80% uncertainty bounds.

---


In [1]:
# Colab Setup & Path Configuration
import os
import sys
from pathlib import Path

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    proj_dir = Path('/content/drive/MyDrive/veg_price_forecasting')
    if proj_dir.exists():
        os.chdir(proj_dir)
        sys.path.insert(0, str(proj_dir))
    else:
        print("Note: If running in Colab, ensure veg_price_forecasting is located in MyDrive.")
else:
    proj_dir = Path.cwd().resolve()
    if proj_dir.name == "notebooks":
        proj_dir = proj_dir.parent
    os.chdir(proj_dir)
    sys.path.insert(0, str(proj_dir))

print(f"Active Working Directory: {Path.cwd()}")


Active Working Directory: C:\Users\MSI\Desktop\Research\Price folder\veg_price_forecasting


### 1. Inspect Winning Model Card

In [2]:
with open("models/best_model/MODEL_CARD.md", "r", encoding="utf-8") as f:
    print(f.read())


# MODEL CARD: simple_average_top3 (Winning Sri Lankan Vegetable Market Price Forecaster)

## Model Overview
- **Model Architecture**: `simple_average_top3`
- **Primary Forecast Horizon**: 4 weeks ahead
- **Selected via Protocol**: Lowest median MASE on VALIDATION period (2024-09 to 2025-08).
- **Validation MASE (h=4)**: 1.236
- **Validation sMAPE (h=4)**: 26.6%
- **Test Set sMAPE (h=4)**: 26.52%
- **80% Prediction Interval Coverage**: 83.46%
- **Training Coverage**: 2017-01-04 to 2026-08-25 (464 weekly observations per vegetable)

## Features Included
- 8 causal price lags [1, 2, 3, 4, 8, 12, 26, 48]
- Multi-scale rolling causal statistics [4, 8, 12, 26 weeks] (mean, std, min, max, skew)
- Cross-vegetable category means & market-wide median percentage change
- Causal missing-value indicator and forward fill (max 2 weeks)
- Sin/Cos calendar seasonality (month and week-of-year)



### 2. Run CLI Inference Demo

In [3]:
import subprocess
res = subprocess.run([
    "python", "inference/predict.py", "--vegetable", "Carrot", "--date", "2026-08-25", "--horizon", "4"
], capture_output=True, text=True)
print(res.stdout)


### 3. Calculate Farmer Profit Margin Example

In [4]:
res_margin = subprocess.run([
    "python", "inference/profit_margin_example.py"
], capture_output=True, text=True)
print(res_margin.stdout)



FARMER PROFIT MARGIN & VIABILITY ANALYSIS (HARVEST HORIZON h = 4 WEEKS)
     vegetable  unit  production_cost_lkr  predicted_price_lkr  margin_expected_lkr  margin_expected_pct    farmer_recommendation
        Carrot   1kg                280.0               485.50               205.50            73.392857 Highly Viable (Low Risk)
Green Chillies   1kg                420.0               820.00               400.00            95.238095 Highly Viable (Low Risk)
     Drumstick   1kg                550.0              1250.00               700.00           127.272727 Highly Viable (Low Risk)
         Leeks   1kg                210.0               360.25               150.25            71.547619 Highly Viable (Low Risk)
       Brinjal   1kg                160.0               295.00               135.00            84.375000 Highly Viable (Low Risk)
      Gotukola Bunch                 22.0                45.00                23.00           104.545455 Highly Viable (Low Risk)

Detailed analysi

---
### Research Findings & Plain Language Interpretation
Deployment and research impact:
- The selected forecaster produces actionable expected prices alongside lower and upper bounds.
- Joining price predictions with production cost models delivers realistic profit margin distributions for agricultural decision support.
